In [1]:
import json
import os 
import glob
from datetime import datetime, timedelta
import matplotlib.pyplot as plt

import torch
from torch.utils.data import Dataset, DataLoader

In [2]:
class LazyDatasetList(Dataset):
    def __init__(self, data_files, transform=None, time_interval=timedelta(minutes=10)):
        self.data_files=data_files
        self.transform = transform
        self.time_interval = time_interval

        # Build a mapping from datetime -> filepath
        self.time_format = "%Y%m%d%H%M"
        self.time_to_file = {}
        for f in self.data_files:
            ts = os.path.splitext(os.path.basename(f))[0]
            try:
                dt = datetime.strptime(ts, self.time_format)
                self.time_to_file[dt] = f
            except ValueError:
                continue  # Skip malformed filenames

        # Sort the datetimes
        sorted_times = sorted(self.time_to_file.keys())

        # Build valid pairs [t-interval, t]
        self.time_list = []
        for t in sorted_times:
            next_t = t - self.time_interval #t-1
            next_t_2 = next_t - self.time_interval #t-2
            next_t_3 = next_t_2 - self.time_interval #t-3
            if next_t in self.time_to_file and next_t_2 in self.time_to_file and next_t_3 in self.time_to_file:
                self.time_list.append((self.time_to_file[t], 
                                       self.time_to_file[next_t], 
                                       self.time_to_file[next_t_2], 
                                       self.time_to_file[next_t_3]))

    def __len__(self):
        return len(self.time_list)

    def __getitem__(self, idx):
        f_t, f_t_1, f_t_2, f_t_3 = self.time_list[idx]
        full_str = f'{os.path.basename(f_t_3)},{os.path.basename(f_t_2)},{os.path.basename(f_t_1)},{os.path.basename(f_t)}'
        # Load data
        data_list = []
        for t_step in [f_t_3, f_t_2, f_t_1, f_t]:
            data=torch.load(t_step).unsqueeze(0)
            if self.transform:
                data = self.transform(data)
            data_list.append(data)
        data=torch.cat(data_list, dim=0)

        return {'data': data, 'source': full_str}


class MinMaxToMinusOneOne:
    def __init__(self, min_vals, max_vals, ):
        self.min_vals = torch.tensor(min_vals).view(-1, 1, 1)
        self.max_vals = torch.tensor(max_vals).view(-1, 1, 1)
        
    def __call__(self, x):
        # x: [C, H, W]
        x = (x - self.min_vals) / (self.max_vals - self.min_vals + 1e-6)  # → [0, 1]
        return x * 2.0 - 1.0  # → [-1, 1]

In [3]:
input_dir='/scratch/nf33/cd3022/pyearth'

# Load saved min/max
with open(os.path.join(input_dir, 'min_max.json'), 'r') as f:
    stats = json.load(f)
    min_vals, max_vals = torch.tensor(stats['min'], dtype=torch.float32), torch.tensor(stats['max'], dtype=torch.float32)

data_files=sorted(glob.glob(os.path.join(input_dir, '*.pt')))
train_dataset = LazyDatasetList(data_files, transform=MinMaxToMinusOneOne(min_vals, max_vals))

/jobfs/163255815.gadi-pbs/ipykernel_4022593/1185948283.py:53: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.min_vals = torch.tensor(min_vals).view(-1, 1, 1)
/jobfs/163255815.gadi-pbs/ipykernel_4022593/1185948283.py:54: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.max_vals = torch.tensor(max_vals).view(-1, 1, 1)


In [4]:
from dataclasses import dataclass
from torchvision import transforms
import torch
import torch.nn as nn
from diffusers import UNet2DModel
from diffusers import DDPMScheduler
import torch.nn.functional as F
from diffusers.optimization import get_cosine_schedule_with_warmup
from diffusers import DDPMPipeline
import os
from accelerate import Accelerator
from tqdm.auto import tqdm
from accelerate import notebook_launcher
from typing import List, Callable, Union, Any, TypeVar, Tuple, Dict
Tensor = TypeVar('torch.tensor')
from abc import abstractmethod

In [5]:
'''
image_size: 64 to 16 (pixel to latents)

'''

@dataclass
class TrainingConfig:
    in_channels=4
    out_channels=1
    image_size = 16  # the generated image resolution
    train_batch_size = 2
    num_epochs = 1
    gradient_accumulation_steps = 100
    learning_rate = 1e-4
    lr_warmup_steps = 600 #5-10% of total
    save_model_epochs = 500
    mixed_precision = "bf16"  # `no` for float32, `fp16` for automatic mixed precision
    output_dir = "/scratch/nf33/cd3022/pyearth/model"  # the model name locally and on the HF Hub
    resume_from_checkpoint = False
    #push_to_hub = True  # whether to upload the saved model to the HF Hub
    #hub_model_id = "<your-username>/<my-awesome-model>"  # the name of the repository to create on the HF Hub
    #hub_private_repo = None
    overwrite_output_dir = True  # overwrite the old model when re-running the notebook
    seed = 0

In [6]:
'''
parse VAE from condition and target latents. All others remain the same

'''
def train_loop(config, 
               model, 
               vae_cond,
               vae_target,
               noise_scheduler, 
               optimizer, 
               train_dataloader, 
               lr_scheduler):

    device = torch.device("cuda")

    # Initialize accelerator and tensorboard logging
    accelerator = Accelerator(
        mixed_precision=config.mixed_precision,
        gradient_accumulation_steps=config.gradient_accumulation_steps,
        log_with="tensorboard",
        project_dir=os.path.join(config.output_dir, "logs"),
    )

    if accelerator.is_main_process:
        if config.output_dir is not None:
            os.makedirs(config.output_dir, exist_ok=True)
        accelerator.init_trackers("train_example")

    '''
    add vae_cond and vae_target to accelerate prepare
    '''
    for param in vae_cond.parameters():
        param.requires_grad = False

    for param in vae_target.parameters():
        param.requires_grad = False
        
    # Prepare everything
    # There is no specific order to remember, you just need to unpack the
    # objects in the same order you gave them to the prepare method.
    model, vae_cond, vae_target, optimizer, train_dataloader, lr_scheduler = accelerator.prepare(
        model, vae_cond, vae_targ, optimizer, train_dataloader, lr_scheduler
    )

    # checkpointing
    checkpt_dir = os.path.join(os.path.join(config.output_dir,  'checkpoint'))
    if not os.path.exists(checkpt_dir):
        os.makedirs(checkpt_dir, exist_ok=True)

    accelerator.register_for_checkpointing(lr_scheduler)

    global_step = 0

    # Now you train the model
    for epoch in range(config.num_epochs):
        progress_bar = tqdm(total=len(train_dataloader), disable=not accelerator.is_local_main_process)
        progress_bar.set_description(f"Epoch {epoch}")

        '''
        encode condition and target into latents.
        '''
        #layer 0-2: conditions layer 2-4: targets
        for step, batch in enumerate(train_dataloader):
            condition_images = batch['data'][:, 0:3, :, :].detach().to(torch.bfloat16).to(device) 
            condition_latents = vae_cond.encode(condition_images)
            clean_images = batch['data'][:, 3:4, :, :].detach().to(torch.bfloat16).to(device) 
            target_latents= vae_target.encode(target_images)
            # Sample noise to add to the images
            noise = torch.randn(target_latents.shape, device=target_latents.device)
            bs = target_latents.shape[0]

            # Sample a random timestep for each image
            timesteps = torch.randint(
                0, noise_scheduler.config.num_train_timesteps, (bs,), device=clean_images.device,
                dtype=torch.int64
            ).to(device)

            # Add noise to the clean images according to the noise magnitude at each timestep
            # (this is the forward diffusion process)
            noisy_images = noise_scheduler.add_noise(target_latents, noise, timesteps)

            with accelerator.accumulate(model):
                # Predict the noise residual

                x_t = torch.cat([noisy_images, condition_latents], dim=1)  
                noise_pred = model(x_t, timesteps, return_dict=False)[0]
                loss = F.mse_loss(noise_pred, noise)

                accelerator.backward(loss)

                if accelerator.sync_gradients:
                    accelerator.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                lr_scheduler.step()
                optimizer.zero_grad()

            progress_bar.update(1)
            logs = {"loss": loss.detach().item(), "lr": lr_scheduler.get_last_lr()[0], "step": global_step}
            progress_bar.set_postfix(**logs)
            accelerator.log(logs, step=global_step)
            global_step += 1

        # After each epoch you optionally sample some demo images with evaluate() and save the model
        if accelerator.is_main_process:
            pipeline = DDPMPipeline(unet=accelerator.unwrap_model(model), scheduler=noise_scheduler)

            #if (epoch + 1) % config.save_image_epochs == 0 or epoch == config.num_epochs - 1:
            #    evaluate(config, epoch, pipeline)

            if (epoch + 1) % config.save_model_epochs == 0 or epoch == config.num_epochs - 1:
                epoch_ckpt_dir = os.path.join(config.output_dir, f"checkpoint/epoch_{epoch+10:04d}")
                accelerator.save_state(epoch_ckpt_dir)
                pipeline.save_pretrained(epoch_ckpt_dir)

In [7]:
class BaseVAE(nn.Module):
    def __init__(self) -> None:
        super(BaseVAE, self).__init__()

    def encode(self, input: Tensor) -> List[Tensor]:
        raise NotImplementedError

    def decode(self, input: Tensor) -> Any:
        raise NotImplementedError

    def sample(self, batch_size:int, current_device: int, **kwargs) -> Tensor:
        raise NotImplementedError

    def generate(self, x: Tensor, **kwargs) -> Tensor:
        raise NotImplementedError

    @abstractmethod
    def forward(self, *inputs: Tensor) -> Tensor:
        pass

    @abstractmethod
    def loss_function(self, *inputs: Any, **kwargs) -> Tensor:
        pass


class VanillaVAE(BaseVAE):
    def __init__(self,
                 input_size: int,
                 in_channels: int,
                 latent_dim: int,
                 hidden_dims: List = None,
                 **kwargs) -> None:
        super(VanillaVAE, self).__init__()

        self.input_size = input_size
        self.latent_dim = latent_dim
        self.in_channels = in_channels
        modules = []
        if hidden_dims is None:
            hidden_dims = [256, 512]

        # Build Encoder
        for h_dim in hidden_dims:
            modules.append(
                nn.Sequential(
                    nn.Conv2d(in_channels, out_channels=h_dim,
                              kernel_size= 3, stride= 2, padding  = 1),
                    nn.GroupNorm(num_groups=max(2, int(h_dim/16)), num_channels=h_dim),
                    nn.LeakyReLU())
            )
            in_channels = h_dim

        
        self.encoder = nn.Sequential(*modules) # [B, hidden_dim[-1], 8, 8]
        self.fc_mu = nn.Conv2d(hidden_dims[-1], latent_dim, 3, 1, 1)
        self.fc_var = nn.Conv2d(hidden_dims[-1], latent_dim, 3, 1, 1)

        # Build Decoder
        modules = []
        self.decoder_input = nn.Conv2d(latent_dim, hidden_dims[-1], 3, 1, 1)

        hidden_dims.reverse()

        for i in range(len(hidden_dims) - 1):
            modules.append(
                nn.Sequential(
                    nn.ConvTranspose2d(hidden_dims[i],
                                       hidden_dims[i + 1],
                                       kernel_size=3,
                                       stride = 2,
                                       padding=1,
                                       output_padding=1),
                    #nn.BatchNorm2d(hidden_dims[i + 1]),
                    nn.GroupNorm(num_groups=max(2, int(hidden_dims[i + 1]/16)), num_channels=hidden_dims[i + 1]),
                    nn.LeakyReLU())
            )

        self.decoder = nn.Sequential(*modules)

        self.final_layer = nn.Sequential(
                            nn.ConvTranspose2d(hidden_dims[-1],
                                               hidden_dims[-1],
                                               kernel_size=3,
                                               stride=2,
                                               padding=1,
                                               output_padding=1),
                            nn.GroupNorm(num_groups=max(2, int(hidden_dims[-1]/16)), num_channels=hidden_dims[-1]),
                            nn.LeakyReLU(),
                            nn.Conv2d(hidden_dims[-1], out_channels= self.in_channels,
                                      kernel_size= 3, padding= 1),
                            nn.Tanh())

    def encode(self, input: Tensor) -> List[Tensor]:
        result = self.encoder(input)

        # Split the result into mu and var components
        # of the latent Gaussian distribution
        mu = self.fc_mu(result)
        log_var = self.fc_var(result)
        return mu, log_var

    def decode(self, z: Tensor) -> Tensor:
        result = self.decoder_input(z)
        result = self.decoder(result)
        result = self.final_layer(result)
        return result

    def reparameterize(self, mu: Tensor, logvar: Tensor) -> Tensor:
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return eps * std + mu

    def forward(self, input: Tensor, **kwargs) -> List[Tensor]:
        mu, log_var = self.encode(input)
        z = self.reparameterize(mu, log_var)
        return  self.decode(z), mu, log_var

    def sample(self,
               num_samples:int,
               current_device: int, **kwargs) -> Tensor:
        z = torch.randn(num_samples,
                        self.latent_dim)

        z = z.to(current_device)

        samples = self.decode(z)
        return samples

    def generate(self, x: Tensor, **kwargs) -> Tensor:
        return self.forward(x)[0]


def vae_loss(recons, input, mu, log_var, **kwargs) -> tuple:
        recons_loss =F.mse_loss(recons, input, reduction='mean')

        kld_loss =-0.5 * torch.mean(1 + log_var - mu.pow(2) - log_var.exp())

        loss = recons_loss + 0.00025* kld_loss
        return loss, recons_loss.detach(), -kld_loss.detach()

In [8]:
config = TrainingConfig()
device = torch.device("cuda")
input_dir='/scratch/nf33/cd3022/pyearth'

# Load saved min/max
with open('/scratch/nf33/cd3022/pyearth/min_max.json', 'r') as f:
    stats = json.load(f)
    min_vals, max_vals = torch.tensor(stats['min'], dtype=torch.bfloat16), torch.tensor(stats['max'], dtype=torch.bfloat16)
    
data_files=glob.glob(os.path.join(input_dir, '*.pt'))
dataset = LazyDatasetList(data_files, transform=MinMaxToMinusOneOne(min_vals, max_vals))
train_dataloader = torch.utils.data.DataLoader(dataset, 
                                                batch_size=config.train_batch_size, 
                                                shuffle=True, 
                                                )
'''
Load VAE
'''
##############################################################################
# CURRENLT NO .../MODEL/VAE/COND... SWAPPING FOR TARGET TO GET SCRIPT WORKING
##############################################################################
vae_output_dir = '/scratch/nf33/cd3022/pyearth/model/vae'
vae_cond = VanillaVAE(input_size=64, in_channels=1, latent_dim=3).to(device)
ckpt_path = os.path.join(vae_output_dir, 'target', "checkpoints/vae_epoch_600.pt")
checkpoint = torch.load(ckpt_path, map_location='cuda' if torch.cuda.is_available() else 'cpu')
vae_cond.load_state_dict(checkpoint['model_state_dict'])
vae_cond.eval()

vae_target = VanillaVAE(input_size=64, in_channels=1, latent_dim=3).to(device)
ckpt_path = os.path.join(vae_output_dir, 'target', "checkpoints/vae_epoch_600.pt")
checkpoint = torch.load(ckpt_path, map_location='cuda' if torch.cuda.is_available() else 'cpu')
vae_target.load_state_dict(checkpoint['model_state_dict'])
vae_target.eval()

# Restore model weights
model = UNet2DModel(
    sample_size=config.image_size,  # the target image resolution
    in_channels=config.in_channels,  # the number of input channels, 3 for RGB images
    out_channels=config.out_channels,  # the number of output channels
    layers_per_block=2,  # how many ResNet layers to use per UNet block
    block_out_channels=(32, 64, 128, 256),  # the number of output channels for each UNet block
    down_block_types=(
        "DownBlock2D",  
        "DownBlock2D",
        "AttnDownBlock2D",
        "DownBlock2D",
    ),
    up_block_types=(
        "UpBlock2D", 
        "AttnUpBlock2D", 
        "UpBlock2D",
        "UpBlock2D",

    ),
).to(device)

noise_scheduler = DDPMScheduler(num_train_timesteps=1000)
optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate)
lr_scheduler = get_cosine_schedule_with_warmup(
    optimizer=optimizer,
    num_warmup_steps=config.lr_warmup_steps,
    num_training_steps=(len(train_dataloader) * config.num_epochs),
)
args = (config, model, vae_cond, vae_target, noise_scheduler, optimizer, train_dataloader, lr_scheduler)

/jobfs/163255815.gadi-pbs/ipykernel_4022593/1185948283.py:53: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.min_vals = torch.tensor(min_vals).view(-1, 1, 1)
/jobfs/163255815.gadi-pbs/ipykernel_4022593/1185948283.py:54: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.max_vals = torch.tensor(max_vals).view(-1, 1, 1)


RuntimeError: Error(s) in loading state_dict for VanillaVAE:
	size mismatch for fc_mu.weight: copying a param with shape torch.Size([1, 512, 3, 3]) from checkpoint, the shape in current model is torch.Size([3, 512, 3, 3]).
	size mismatch for fc_mu.bias: copying a param with shape torch.Size([1]) from checkpoint, the shape in current model is torch.Size([3]).
	size mismatch for fc_var.weight: copying a param with shape torch.Size([1, 512, 3, 3]) from checkpoint, the shape in current model is torch.Size([3, 512, 3, 3]).
	size mismatch for fc_var.bias: copying a param with shape torch.Size([1]) from checkpoint, the shape in current model is torch.Size([3]).
	size mismatch for decoder_input.weight: copying a param with shape torch.Size([512, 1, 3, 3]) from checkpoint, the shape in current model is torch.Size([512, 3, 3, 3]).

# Removed conditional VAE

In [9]:
def train_loop(config, 
               model, 
               vae_target,
               noise_scheduler, 
               optimizer, 
               train_dataloader, 
               lr_scheduler):

    device = torch.device("cuda")

    # Initialize accelerator and tensorboard logging
    accelerator = Accelerator(
        mixed_precision=config.mixed_precision,
        gradient_accumulation_steps=config.gradient_accumulation_steps,
        log_with="tensorboard",
        project_dir=os.path.join(config.output_dir, "logs"),
    )

    if accelerator.is_main_process:
        if config.output_dir is not None:
            os.makedirs(config.output_dir, exist_ok=True)
        accelerator.init_trackers("train_example")

    '''
    add vae_target to accelerate prepare
    '''

    for param in vae_target.parameters():
        param.requires_grad = False
        
    # Prepare everything
    # There is no specific order to remember, you just need to unpack the
    # objects in the same order you gave them to the prepare method.
    model, vae_target, optimizer, train_dataloader, lr_scheduler = accelerator.prepare(
        model, vae_targ, optimizer, train_dataloader, lr_scheduler
    )

    # checkpointing
    checkpt_dir = os.path.join(os.path.join(config.output_dir,  'checkpoint'))
    if not os.path.exists(checkpt_dir):
        os.makedirs(checkpt_dir, exist_ok=True)

    accelerator.register_for_checkpointing(lr_scheduler)

    global_step = 0

    # Now you train the model
    for epoch in range(config.num_epochs):
        progress_bar = tqdm(total=len(train_dataloader), disable=not accelerator.is_local_main_process)
        progress_bar.set_description(f"Epoch {epoch}")

        '''
        encode condition and target into latents.
        '''
        #layer 0-2: conditions layer 2-4: targets
        for step, batch in enumerate(train_dataloader):
            clean_images = batch['data'][:, 3:4, :, :].detach().to(torch.bfloat16).to(device) 
            target_latents= vae_target.encode(target_images)
            # Sample noise to add to the images
            noise = torch.randn(target_latents.shape, device=target_latents.device)
            bs = target_latents.shape[0]

            # Sample a random timestep for each image
            timesteps = torch.randint(
                0, noise_scheduler.config.num_train_timesteps, (bs,), device=clean_images.device,
                dtype=torch.int64
            ).to(device)

            # Add noise to the clean images according to the noise magnitude at each timestep
            # (this is the forward diffusion process)
            noisy_images = noise_scheduler.add_noise(target_latents, noise, timesteps)

            with accelerator.accumulate(model):
                # Predict the noise residual

                x_t = torch.cat(noisy_images, dim=1)  
                noise_pred = model(x_t, timesteps, return_dict=False)[0]
                loss = F.mse_loss(noise_pred, noise)

                accelerator.backward(loss)

                if accelerator.sync_gradients:
                    accelerator.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                lr_scheduler.step()
                optimizer.zero_grad()

            progress_bar.update(1)
            logs = {"loss": loss.detach().item(), "lr": lr_scheduler.get_last_lr()[0], "step": global_step}
            progress_bar.set_postfix(**logs)
            accelerator.log(logs, step=global_step)
            global_step += 1

        # After each epoch you optionally sample some demo images with evaluate() and save the model
        if accelerator.is_main_process:
            pipeline = DDPMPipeline(unet=accelerator.unwrap_model(model), scheduler=noise_scheduler)

            #if (epoch + 1) % config.save_image_epochs == 0 or epoch == config.num_epochs - 1:
            #    evaluate(config, epoch, pipeline)

            if (epoch + 1) % config.save_model_epochs == 0 or epoch == config.num_epochs - 1:
                epoch_ckpt_dir = os.path.join(config.output_dir, f"checkpoint/epoch_{epoch+10:04d}")
                accelerator.save_state(epoch_ckpt_dir)
                pipeline.save_pretrained(epoch_ckpt_dir)

In [12]:
config = TrainingConfig()
device = torch.device("cuda")
input_dir='/scratch/nf33/cd3022/pyearth'

# Load saved min/max
with open('/scratch/nf33/cd3022/pyearth/min_max.json', 'r') as f:
    stats = json.load(f)
    min_vals, max_vals = torch.tensor(stats['min'], dtype=torch.bfloat16), torch.tensor(stats['max'], dtype=torch.bfloat16)
    
data_files=glob.glob(os.path.join(input_dir, '*.pt'))
dataset = LazyDatasetList(data_files, transform=MinMaxToMinusOneOne(min_vals, max_vals))
train_dataloader = torch.utils.data.DataLoader(dataset, 
                                                batch_size=config.train_batch_size, 
                                                shuffle=True, 
                                                )
'''
Load VAE
'''
##############################################################################
# CURRENLT NO .../MODEL/VAE/COND... SWAPPING FOR TARGET TO GET SCRIPT WORKING
##############################################################################
vae_output_dir = '/scratch/nf33/cd3022/pyearth/model/vae'

vae_target = VanillaVAE(input_size=64, in_channels=1, latent_dim=1).to(device)
ckpt_path = os.path.join(vae_output_dir, 'target', "checkpoints/vae_epoch_600.pt")
checkpoint = torch.load(ckpt_path, map_location='cuda' if torch.cuda.is_available() else 'cpu')
vae_target.load_state_dict(checkpoint['model_state_dict'])
vae_target.eval()

# Restore model weights
model = UNet2DModel(
    sample_size=config.image_size,  # the target image resolution
    in_channels=config.in_channels,  # the number of input channels, 3 for RGB images
    out_channels=config.out_channels,  # the number of output channels
    layers_per_block=2,  # how many ResNet layers to use per UNet block
    block_out_channels=(32, 64, 128, 256),  # the number of output channels for each UNet block
    down_block_types=(
        "DownBlock2D",  
        "DownBlock2D",
        "AttnDownBlock2D",
        "DownBlock2D",
    ),
    up_block_types=(
        "UpBlock2D", 
        "AttnUpBlock2D", 
        "UpBlock2D",
        "UpBlock2D",

    ),
).to(device)

noise_scheduler = DDPMScheduler(num_train_timesteps=1000)
optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate)
lr_scheduler = get_cosine_schedule_with_warmup(
    optimizer=optimizer,
    num_warmup_steps=config.lr_warmup_steps,
    num_training_steps=(len(train_dataloader) * config.num_epochs),
)
args = (config, model, vae_target, noise_scheduler, optimizer, train_dataloader, lr_scheduler)

/jobfs/163255815.gadi-pbs/ipykernel_4022593/1185948283.py:53: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.min_vals = torch.tensor(min_vals).view(-1, 1, 1)
/jobfs/163255815.gadi-pbs/ipykernel_4022593/1185948283.py:54: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.max_vals = torch.tensor(max_vals).view(-1, 1, 1)


In [13]:
args

(TrainingConfig(),
 UNet2DModel(
   (conv_in): Conv2d(4, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
   (time_proj): Timesteps()
   (time_embedding): TimestepEmbedding(
     (linear_1): Linear(in_features=32, out_features=128, bias=True)
     (act): SiLU()
     (linear_2): Linear(in_features=128, out_features=128, bias=True)
   )
   (down_blocks): ModuleList(
     (0): DownBlock2D(
       (resnets): ModuleList(
         (0-1): 2 x ResnetBlock2D(
           (norm1): GroupNorm(32, 32, eps=1e-05, affine=True)
           (conv1): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
           (time_emb_proj): Linear(in_features=128, out_features=32, bias=True)
           (norm2): GroupNorm(32, 32, eps=1e-05, affine=True)
           (dropout): Dropout(p=0.0, inplace=False)
           (conv2): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
           (nonlinearity): SiLU()
         )
       )
       (downsamplers): ModuleList(
         (0): Downsamp